In [15]:
import pandas as pd
import numpy as np

def build_rolling_classification_dataset(
    df: pd.DataFrame,
    window: int = 28,
    step: int = 7,                 # step=7 gives weekly sliding windows 
    horizon_window: int = 28,      # future window length for label
    increase_threshold: float = 0.05,  # 5% threshold for "increase" label (balanced)
    eps: float = 1e-6
):
    """
    Build supervised dataset for demand-direction classification (per Category).
    
    Target (binary):
      y=1 if mean_future > mean_current * (1 + increase_threshold)
      y=0 otherwise
    
    Returns:
      X: DataFrame of features
      y: Series target labels
      meta: DataFrame with CategoryName, window_start, window_end (for time-based split)
    """

    data = df.copy()
    data["OrderDate"] = pd.to_datetime(data["OrderDate"], errors="coerce")
    data = data.dropna(subset=["OrderDate"])

    # Shipped only
    data = data[data["Status"].astype(str).str.lower() == "shipped"].copy()

    # numeric qty
    data["OrderItemQuantity"] = pd.to_numeric(data["OrderItemQuantity"], errors="coerce").fillna(0)

    if data.empty:
        raise ValueError("No shipped records after cleaning.")

    # daily category demand
    daily = (
        data.groupby(["OrderDate", "CategoryName"], as_index=False)["OrderItemQuantity"]
            .sum()
            .rename(columns={"OrderItemQuantity": "demand"})
    )

    min_date = daily["OrderDate"].min()
    max_date = daily["OrderDate"].max()
    full_dates = pd.date_range(min_date, max_date, freq="D")

    X_rows = []
    y_rows = []
    meta_rows = []

    categories = daily["CategoryName"].unique()

    for cat in categories:
        s = daily[daily["CategoryName"] == cat].set_index("OrderDate")[["demand"]]
        s = s.reindex(full_dates, fill_value=0).reset_index().rename(columns={"index": "OrderDate"})
        s["CategoryName"] = cat

        # Need current window + future window
        total_needed = window + horizon_window
        if len(s) < total_needed:
            continue

        # rolling start positions
        for start_i in range(0, len(s) - total_needed + 1, step):
            cur = s.iloc[start_i:start_i + window]["demand"]
            fut = s.iloc[start_i + window:start_i + window + horizon_window]["demand"]

            cur_mean = cur.mean()
            fut_mean = fut.mean()

            # label with threshold
            y = 1 if fut_mean > cur_mean * (1 + increase_threshold) else 0

            # features (current window)
            cur_sum = cur.sum()
            cur_std = cur.std(ddof=0)
            coverage = (cur > 0).mean()
            volatility_ratio = cur_std / (cur_mean + eps)

            # simple trend inside current window: compare first half vs second half
            half = window // 2
            first_half_mean = cur.iloc[:half].mean()
            second_half_mean = cur.iloc[half:].mean()
            intra_growth = (second_half_mean - first_half_mean) / (first_half_mean + eps)

            # future activity indicator (not leaking target, but helps handle all-zero futures)
            # (We keep it OUT to avoid leakage. So we do NOT include future stats as features.)

            X_rows.append({
                "CategoryName": cat,
                "cur_mean": float(cur_mean),
                "cur_sum": float(cur_sum),
                "cur_std": float(cur_std),
                "coverage": float(coverage),
                "volatility_ratio": float(volatility_ratio),
                "intra_growth": float(intra_growth),
            })

            y_rows.append(y)

            meta_rows.append({
                "CategoryName": cat,
                "window_start": s.iloc[start_i]["OrderDate"],
                "window_end": s.iloc[start_i + window - 1]["OrderDate"],
                "future_start": s.iloc[start_i + window]["OrderDate"],
                "future_end": s.iloc[start_i + total_needed - 1]["OrderDate"]
            })

    X = pd.DataFrame(X_rows)
    y = pd.Series(y_rows, name="target_up")
    meta = pd.DataFrame(meta_rows)

    # Remove CategoryName from X if you want pure numeric features
    # (Keep it if you plan one-hot encoding)
    return X, y, meta

In [16]:
import pandas as pd
from pathlib import Path

base_df = pd.read_csv("ML-Dataset.csv")
base_df["OrderDate"] = pd.to_datetime(base_df["OrderDate"], errors="coerce")
base_df["OrderItemQuantity"] = pd.to_numeric(base_df["OrderItemQuantity"], errors="coerce")

feedback_path = Path("feedback/feedback_log.csv")

if feedback_path.exists():
    feedback_df = pd.read_csv(feedback_path).reset_index(drop=True)

    last_base_date = base_df["OrderDate"].max()
    feedback_df["aligned_date"] = [
        last_base_date + pd.Timedelta(days=i + 1) for i in range(len(feedback_df))
    ]

    feedback_transformed = pd.DataFrame({
        "OrderDate": feedback_df["aligned_date"],
        "ProductName": feedback_df["product_name"],
        "CategoryName": feedback_df["category_name"],
        "OrderItemQuantity": pd.to_numeric(feedback_df["actual_sales"], errors="coerce"),
        "Status": "Shipped"
    })

    feedback_transformed = feedback_transformed.dropna(
        subset=["OrderDate", "ProductName", "CategoryName", "OrderItemQuantity"]
    )

    needed_cols = ["OrderDate", "ProductName", "CategoryName", "OrderItemQuantity", "Status"]

    for col in needed_cols:
        if col not in base_df.columns:
            base_df[col] = pd.NA

    updated_df = pd.concat(
        [base_df[needed_cols], feedback_transformed[needed_cols]],
        ignore_index=True
    )

    print("Base rows:", len(base_df))
    print("Feedback rows added:", len(feedback_transformed))
    print("Merged rows:", len(updated_df))

else:
    updated_df = base_df.copy()

Base rows: 400
Feedback rows added: 1
Merged rows: 401


/var/folders/nc/364srb_x0m926dqwk2stdknw0000gn/T/ipykernel_4156/457677093.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  base_df["OrderDate"] = pd.to_datetime(base_df["OrderDate"], errors="coerce")


In [17]:
print(updated_df["OrderDate"].min())
print(updated_df["OrderDate"].max())
print(updated_df["OrderDate"].nunique())

2013-06-21 00:00:00
2017-11-02 00:00:00
76


In [27]:
updated_df.to_csv("updated_base_history.csv", index=False)

In [18]:
X, y, meta = build_rolling_classification_dataset(updated_df, window=28, step=7)
X["intra_growth"] = X["intra_growth"].clip(-5, 5)
X = X.drop(columns=["CategoryName"], errors="ignore")

In [19]:
print("Samples:", len(X))
print("Positive rate:", y.mean())
meta["CategoryName"].value_counts()

Samples: 1105
Positive rate: 0.20361990950226244


CategoryName
Storage         221
CPU             221
Video Card      221
Mother Board    221
RAM             221
Name: count, dtype: int64

In [20]:
meta["CategoryName"].value_counts()

CategoryName
Storage         221
CPU             221
Video Card      221
Mother Board    221
RAM             221
Name: count, dtype: int64

In [21]:
y.value_counts(normalize=True)

target_up
0    0.79638
1    0.20362
Name: proportion, dtype: float64

In [22]:
X.describe()

,cur_mean,cur_sum,cur_std,coverage,volatility_ratio,intra_growth
count,1105.000000,1105.000000,1105.000000,1105.000000,1105.000000,1105.000000
mean,2.056335,57.577376,7.968078,0.017744,1.239973,0.462699
std,4.391594,122.964638,15.448658,0.033722,2.033981,1.642345
min,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2.178571,61.000000,11.320189,0.035714,3.066748,0.000000
max,31.071429,870.000000,95.288258,0.178571,5.196152,5.000000


In [23]:
X.select_dtypes(include=[np.number]).corr()

,cur_mean,cur_sum,cur_std,coverage,volatility_ratio,intra_growth
cur_mean,1.000000,1.000000,0.968872,0.914116,0.607135,0.270690
cur_sum,1.000000,1.000000,0.968872,0.914116,0.607135,0.270690
cur_std,0.968872,0.968872,1.000000,0.865671,0.726317,0.341297
coverage,0.914116,0.914116,0.865671,1.000000,0.686893,0.287915
volatility_ratio,0.607135,0.607135,0.726317,0.686893,1.000000,0.477062
intra_growth,0.270690,0.270690,0.341297,0.287915,0.477062,1.000000


In [24]:
print(X.columns)

Index(['cur_mean', 'cur_sum', 'cur_std', 'coverage', 'volatility_ratio',
       'intra_growth'],
      dtype='object')


In [25]:
full_dataset = pd.concat([meta, X, y], axis=1)
full_dataset.to_csv("rolling_supervised_dataset_v2.csv", index=False)

In [26]:
print(meta.head())
print(meta.tail())

  CategoryName window_start window_end future_start future_end
0      Storage   2013-06-21 2013-07-18   2013-07-19 2013-08-15
1      Storage   2013-06-28 2013-07-25   2013-07-26 2013-08-22
2      Storage   2013-07-05 2013-08-01   2013-08-02 2013-08-29
3      Storage   2013-07-12 2013-08-08   2013-08-09 2013-09-05
4      Storage   2013-07-19 2013-08-15   2013-08-16 2013-09-12
     CategoryName window_start window_end future_start future_end
1100          RAM   2017-08-11 2017-09-07   2017-09-08 2017-10-05
1101          RAM   2017-08-18 2017-09-14   2017-09-15 2017-10-12
1102          RAM   2017-08-25 2017-09-21   2017-09-22 2017-10-19
1103          RAM   2017-09-01 2017-09-28   2017-09-29 2017-10-26
1104          RAM   2017-09-08 2017-10-05   2017-10-06 2017-11-02
